# Test Snowflake UDF: PROCESS_FILE_UDF
Ce notebook teste la UDF `PROCESS_FILE_UDF` sur un fichier audio present dans un stage.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

# Parametres de test
STAGE_PATH = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_SOUNDS/asthma/"
CLASS_NAME = "Asthma"
AUDIO_FILE = None  # ex: 'patient_001.wav' ; None = auto-pick premier .wav

print("Session OK")
print("Stage :", STAGE_PATH)
print("Class :", CLASS_NAME)

In [ ]:
# Liste les fichiers du stage et choisit un fichier audio
ls_df = session.sql(f"LIST {STAGE_PATH}").to_pandas()

if ls_df.empty:
    raise ValueError(f"Aucun fichier trouve dans {STAGE_PATH}")

ls_df["FILE_NAME"] = ls_df["name"].str.split('/').str[-1]
audio_candidates = ls_df[ls_df["FILE_NAME"].str.lower().str.endswith(".wav", na=False)]

if audio_candidates.empty:
    raise ValueError("Aucun .wav trouve dans le stage")

if AUDIO_FILE is None:
    AUDIO_FILE = audio_candidates.iloc[0]["FILE_NAME"]

print("Fichier teste :", AUDIO_FILE)
audio_candidates[["FILE_NAME", "size", "last_modified"]].head(10)

In [ ]:
# Pre-check: contexte SQL + existence UDF
ctx = session.sql("""
    SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH
""").to_pandas()
print(ctx.to_string(index=False))

udf_check = session.sql("""
    SHOW USER FUNCTIONS LIKE 'PROCESS_FILE_UDF' IN SCHEMA M2_ISD_EQUIPE_1_DB.PROCESSED
""").to_pandas()

print(f"UDFs trouvees: {len(udf_check)}")
print("Colonnes retournees:", list(udf_check.columns))

if not udf_check.empty:
    # Les noms de colonnes de SHOW peuvent varier selon les versions Snowflake.
    col_map = {str(c).lower(): c for c in udf_check.columns}
    preferred = ["name", "schema_name", "arguments", "language"]
    available = [col_map[c] for c in preferred if c in col_map]

    if available:
        display(udf_check[available])
    else:
        display(udf_check)
else:
    print("PROCESS_FILE_UDF non trouvee dans M2_ISD_EQUIPE_1_DB.PROCESSED")

In [ ]:
# Appel de la UDF avec diagnostic d'erreur complet
def sql_escape(s: str) -> str:
    return s.replace("'", "''")

audio_file_sql = sql_escape(AUDIO_FILE)
stage_sql = sql_escape(STAGE_PATH)
class_sql = sql_escape(CLASS_NAME)

# Appel fully-qualified pour eviter les problemes de schema courant
query = f"""
SELECT M2_ISD_EQUIPE_1_DB.PROCESSED.PROCESS_FILE_UDF(
  '{audio_file_sql}',
  '{stage_sql}',
  '{class_sql}'
) AS RESULT
"""

print("Executing SQL:\n", query)

try:
    rows = session.sql(query).collect()
    if not rows:
        raise RuntimeError("La requete a retourne 0 ligne")

    result = rows[0]["RESULT"]
    print("Result STATUS:", result.get("STATUS"))
    print("Result ACTION:", result.get("ACTION"))
    result
except Exception as e:
    print("Echec appel UDF")
    print("Type:", type(e).__name__)
    print("Message:", str(e))
    
    # Aide de debug immediate
    print("\nChecks rapides:")
    print("1) La UDF existe dans M2_ISD_EQUIPE_1_DB.PROCESSED")
    print("2) Le fichier et le stage sont corrects")
    print("3) Le role a USAGE sur DB/SCHEMA et READ sur stage")
    raise

In [ ]:
# Verification des inserts metadata (selon ton implentation actuelle)
candidate_tables = [
    "M2_ISD_EQUIPE_1_DB.TEST.USER_RESPIRATORY_SOUNDS_METADATA",
    "M2_ISD_EQUIPE_1_DB.PROCESSED.RESPIRATORY_SOUNDS_METADATA",
]

for table_name in candidate_tables:
    try:
        df = session.sql(f"""
            SELECT *
            FROM {table_name}
            WHERE FILE_NAME = '{audio_file_sql}'
            ORDER BY CREATED_AT DESC
            LIMIT 5
        """).to_pandas()
        print(f"\nTable testee: {table_name} -> {len(df)} ligne(s)")
        if not df.empty:
            display(df)
    except Exception as e:
        print(f"Table non accessible: {table_name} ({e})")